In [4]:
import pandas as pd
import numpy as np

# Loading the raw tox_khan file and previewing it
tox_khan = pd.read_csv("../data/01_raw/tox_khan.csv")
tox_khan.head()

,SMILES,Target
0,O=C(C1CCCCC1)N1CC(=O)N2CCc3ccccc3C2C1,1
1,CC1CCc2cc(F)cc3c(=O)c(C(=O)O)cn1c23,1
2,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,1
3,C#CC1(OC(N)=O)CCCCC1,0
4,O=P(O)(O)C(O)(Cc1cccnc1)P(=O)(O)O,0


In [8]:
from rdkit.Chem import AllChem as Chem 
from rdkit.Chem import Descriptors as Descs

def get_descriptors(smile):
    '''Generates the full list of molecular descriptors for a molecule from its SMILES'''
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None

    # Sanitizes and adds hydrogens to the molecules
    Chem.SanitizeMol(mol, catchErrors=True)
    Chem.AddHs(mol)

    # Calculates all Descriptors for a molecule and appends them to a list
    all_desc = []
    descriptors = Descs.CalcMolDescriptors(mol, list(Descs.descList))
    all_desc.append(descriptors)

    # Creates a dataframe for all descriptors of a molecule and returns it
    desc_df = pd.DataFrame(all_desc, columns=[desc[0] for desc in Descs.descList])
    return desc_df

# Generates the descriptors for each SMILE in the Tox Khan dataset and drops any rows where the SMILES didn't parse
khan_descs = tox_khan["SMILES"].apply(get_descriptors)
khan_descs_df = pd.concat(khan_descs.to_list()).reset_index(drop=True)
print(f"Descriptors generated successfully: {khan_descs_df.shape[1]} descriptors for {khan_descs_df.shape[0]} molecules")

tox_khan_descs = pd.concat([tox_khan, khan_descs_df],axis=1)
tox_khan_descs.dropna(inplace=True)
print(f"Descriptor set fully generated successfully: {tox_khan_descs.shape[1] - 2} descriptors for {tox_khan_descs.shape[0]} molecules")

tox_khan_descs.head()

Descriptors generated successfully: 217 descriptors for 2588 molecules
Descriptor set fully generated successfully: 217 descriptors for 2584 molecules


,SMILES,Target,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,O=C(C1CCCCC1)N1CC(=O)N2CCc3ccccc3C2C1,1,12.840378,12.840378,0.049760,0.049760,0.799367,25.043478,312.413,288.221,...,0,0,0,0,0,0,0,0,0,0
1,CC1CCc2cc(F)cc3c(=O)c(C(=O)O)cn1c23,1,13.554306,13.554306,0.096446,-1.278486,0.856751,17.684211,261.252,249.156,...,0,0,0,0,0,0,0,0,0,0
2,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3c(CN(C)C)c(O...,1,13.230323,13.230323,0.104072,-1.840670,0.486542,19.322581,421.453,398.269,...,0,0,0,0,0,0,0,0,0,0
3,C#CC1(OC(N)=O)CCCCC1,0,10.536064,10.536064,0.704282,-0.768657,0.600262,20.916667,167.208,154.104,...,0,0,0,1,0,0,0,0,0,0
4,O=P(O)(O)C(O)(Cc1cccnc1)P(=O)(O)O,0,11.036035,11.036035,0.098750,-5.410055,0.469630,13.705882,283.113,272.025,...,0,0,0,0,0,0,0,0,0,0


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold

# Separate the features and target variable
X_desc = tox_khan_descs.drop(columns=["SMILES", "Target", "Ipc"])
y = tox_khan_descs["Target"]

# Identify binary and numerical columns for preprocessing and standardization
binary_cols = [col for col in X_desc.columns if X_desc[col].nunique == 2]
numerical_cols = [col for col in X_desc.columns if col not in binary_cols]

# Create a pipeline for preprocessing and standardization by removing non-varied features and scaling the numerical features
numeric_pipeline = Pipeline([
    ('remove_constants', VarianceThreshold(threshold=0.0)),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('binary_features', 'passthrough', binary_cols),
        ('numerical_features', numeric_pipeline, numerical_cols)
    ],
    remainder='drop'
)

preprocessor.set_output(transform='pandas')

# Fit the preprocessing pipeline to the data and transform the features
X_desc_processed = preprocessor.fit_transform(X_desc)

X_desc_processed.head()

,numerical_features__MaxAbsEStateIndex,numerical_features__MaxEStateIndex,numerical_features__MinAbsEStateIndex,numerical_features__MinEStateIndex,numerical_features__qed,numerical_features__SPS,numerical_features__MolWt,numerical_features__HeavyAtomMolWt,numerical_features__ExactMolWt,numerical_features__NumValenceElectrons,...,numerical_features__fr_quatN,numerical_features__fr_sulfide,numerical_features__fr_sulfonamd,numerical_features__fr_sulfone,numerical_features__fr_term_acetylene,numerical_features__fr_tetrazole,numerical_features__fr_thiazole,numerical_features__fr_thiophene,numerical_features__fr_unbrch_alkane,numerical_features__fr_urea
0,0.696280,0.696280,-0.461606,0.696036,1.225442,0.415130,-0.157087,-0.170700,-0.156600,-0.110597,...,-0.123191,-0.236684,-0.196439,-0.092666,-0.090518,-0.102758,-0.133974,-0.12804,-0.180955,-0.161635
1,0.929915,0.929915,-0.356513,-0.212719,1.488883,-0.179280,-0.291505,-0.280684,-0.290954,-0.272705,...,-0.123191,-0.236684,-0.196439,-0.092666,-0.090518,-0.102758,-0.133974,-0.12804,-0.180955,-0.161635
2,0.823890,0.823890,-0.339344,-0.597351,-0.210691,-0.046948,0.129399,0.139130,0.129914,0.146074,...,-0.123191,-0.236684,-0.196439,-0.092666,-0.090518,-0.102758,-0.133974,-0.12804,-0.180955,-0.161635
3,-0.057815,-0.057815,1.011767,0.136094,0.311378,0.081807,-0.538592,-0.548295,-0.538047,-0.488849,...,-0.123191,-0.236684,-0.196439,-0.092666,11.047516,-0.102758,-0.133974,-0.12804,-0.180955,-0.161635
4,0.105802,0.105802,-0.351325,-3.039442,-0.288329,-0.500610,-0.234069,-0.216299,-0.233323,-0.286214,...,-0.123191,-0.236684,-0.196439,-0.092666,-0.090518,-0.102758,-0.133974,-0.12804,-0.180955,-0.161635


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
